In [1]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# ---------------------------
# Dati: BCI-IV-2b (S01–S09) come nella tua tabella
# ---------------------------
data = {
    "EEGNet": {
        "None": [98.58, 99.10, 98.19, 84.44, 92.95, 99.53, 97.41, 98.91, 93.51],
        "RDWT": [99.29, 100.00, 98.55, 80.00, 92.31, 100.00, 97.04, 98.19, 99.13],
    },
    "ShallowConvNet": {
        "None": [99.65, 99.10, 97.83, 82.22, 91.67, 99.06, 95.19, 99.28, 98.70],
        "RDWT": [99.65, 99.10, 96.74, 86.67, 91.67, 99.06, 93.33, 98.55, 98.70],
    },
    "MBEEG_SENet": {
        "None": [98.94, 98.20, 99.28, 82.22, 92.95, 100.00, 97.04, 98.91, 100.00],
        "RDWT": [98.94, 99.10, 97.83, 84.44, 92.95, 100.00, 97.04, 98.91, 99.57],
    },
    "EEGTCNet": {
        "None": [98.23, 99.10, 97.10, 84.44, 90.38, 100.00, 97.41, 98.19, 97.40],
        "RDWT": [99.65, 98.20, 98.19, 82.22, 91.67, 99.53, 97.04, 99.64, 98.70],
    },
}

def holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.zeros(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order, start=1):
        adj_p = (m - rank + 1) * pvals[idx]
        running = max(running, adj_p)  # step-down monotono
        adj[idx] = running
    return np.minimum(adj, 1.0)

def wilcoxon_one_sided(a, b):
    """Wilcoxon signed-rank appaiato, one-sided H1: RDWT > None.
       Usa 'exact' se disponibile, altrimenti fallback."""
    try:
        return wilcoxon(b, a, alternative="greater", zero_method="wilcox", method="exact").pvalue
    except TypeError:
        return wilcoxon(b, a, alternative="greater", zero_method="wilcox").pvalue

rows = []
for model, vals in data.items():
    a, b = vals["None"], vals["RDWT"]
    p = wilcoxon_one_sided(a, b)
    avg_none = float(np.mean(a))
    avg_rdwt = float(np.mean(b))
    rows.append({
        "Model": model,
        "Avg_None": avg_none,
        "Avg_RDWT": avg_rdwt,
        "Delta_Avg_pp": avg_rdwt - avg_none,
        "p_wilcoxon_one_sided": p,
    })

df = pd.DataFrame(rows)
df["p_Holm"] = holm_bonferroni(df["p_wilcoxon_one_sided"].values)

# Stampa risultati da incollare nella tabella/nota
pd.set_option("display.precision", 4)
print(df[["Model","Delta_Avg_pp","p_wilcoxon_one_sided","p_Holm"]])

# (Opzionale) righe commento LaTeX per riferimento
print("\n% LaTeX: p-value Wilcoxon (paired, one-sided; Holm-corrected tra 4 modelli)")
for _, r in df.iterrows():
    print(f"% {r['Model']}: p = {r['p_wilcoxon_one_sided']:.4f}  (Holm {r['p_Holm']:.4f})")


            Model  Delta_Avg_pp  p_wilcoxon_one_sided  p_Holm
0          EEGNet        0.2100                0.4102  1.0000
1  ShallowConvNet        0.0856                0.6425  1.0000
2     MBEEG_SENet        0.1378                0.3575  1.0000
3        EEGTCNet        0.2878                0.2129  0.8516

% LaTeX: p-value Wilcoxon (paired, one-sided; Holm-corrected tra 4 modelli)
% EEGNet: p = 0.4102  (Holm 1.0000)
% ShallowConvNet: p = 0.6425  (Holm 1.0000)
% MBEEG_SENet: p = 0.3575  (Holm 1.0000)
% EEGTCNet: p = 0.2129  (Holm 0.8516)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:198: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:198: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# ---------------------------
# Dati BCI-IV-2b (S01–S09) come nella tabella
# ---------------------------
data = {
    "EEGNet": {
        "None": [98.58, 99.10, 98.19, 84.44, 92.95, 99.53, 97.41, 98.91, 93.51],
        "RDWT": [99.29, 100.00, 98.55, 80.00, 92.31, 100.00, 97.04, 98.19, 99.13],
    },
    "ShallowConvNet": {
        "None": [99.65, 99.10, 97.83, 82.22, 91.67, 99.06, 95.19, 99.28, 98.70],
        "RDWT": [99.65, 99.10, 96.74, 86.67, 91.67, 99.06, 93.33, 98.55, 98.70],
    },
    "MBEEG_SENet": {
        "None": [98.94, 98.20, 99.28, 82.22, 92.95, 100.00, 97.04, 98.91, 100.00],
        "RDWT": [98.94, 99.10, 97.83, 84.44, 92.95, 100.00, 97.04, 98.91, 99.57],
    },
    "EEGTCNet": {
        "None": [98.23, 99.10, 97.10, 84.44, 90.38, 100.00, 97.41, 98.19, 97.40],
        "RDWT": [99.65, 98.20, 98.19, 82.22, 91.67, 99.53, 97.04, 99.64, 98.70],
    },
}

def wilcoxon_one_sided(a, b):
    """Wilcoxon signed-rank paired, one-sided (H1: RDWT > None).
       method='auto' per replicare i p-value che ottieni di default."""
    try:
        return wilcoxon(b, a, alternative="greater", zero_method="wilcox", method="auto").pvalue
    except TypeError:
        # SciPy < 1.11 non ha 'method'
        return wilcoxon(b, a, alternative="greater", zero_method="wilcox").pvalue

rows = []
for model, vals in data.items():
    p = wilcoxon_one_sided(vals["None"], vals["RDWT"])
    rows.append({"Model": model, "p_value": p})

df = pd.DataFrame(rows)
pd.set_option("display.precision", 4)
print(df)

# --- Snippet LaTeX: da appendere SOLO alle righe RDWT come ultima colonna 'p' ---
print("\n% Incolla questi p-value come ultima colonna 'p' nelle righe RDWT (una sola colonna aggiuntiva):")
for _, r in df.iterrows():
    print(f"% {r['Model']} RDWT: & {r['p_value']:.4f} \\\\")



            Model  p_value
0          EEGNet   0.4102
1  ShallowConvNet   0.6425
2     MBEEG_SENet   0.3575
3        EEGTCNet   0.2129

% Incolla questi p-value come ultima colonna 'p' nelle righe RDWT (una sola colonna aggiuntiva):
% EEGNet RDWT: & 0.4102 \\
% ShallowConvNet RDWT: & 0.6425 \\
% MBEEG_SENet RDWT: & 0.3575 \\
% EEGTCNet RDWT: & 0.2129 \\


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:198: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
